In [1]:
"""
S01 baseline overflight — integrate notebooks 01–06.

Scenario:
  - One full polar overflight with seeded clouds over the target corridor (notebook 02)
  - 50-target meridian grid (notebook 01)
  - Nadir approach until lead margin before each target, then OBC engage
  - SequentialTargetBaselinePolicy (bearing-angle observations; not direct actuation)
  - Take-picture on every target; memory budget 10 (notebook 06)

Video: torque + pointing telemetry, cumulative latent capture reward, shutter markers.
Verification: s01_utils/baseline_overflight.py
Export: artifacts/07-baseline-overflight.mp4
"""

'\nS01 baseline overflight — integrate notebooks 01–06.\n\nScenario:\n  - One full polar overflight with seeded clouds over the target corridor (notebook 02)\n  - 50-target meridian grid (notebook 01)\n  - Nadir approach until lead margin before each target, then OBC engage\n  - SequentialTargetBaselinePolicy (bearing-angle observations; not direct actuation)\n  - Take-picture on every target; memory budget 10 (notebook 06)\n\nVideo: torque + pointing telemetry, cumulative latent capture reward, shutter markers.\nVerification: s01_utils/baseline_overflight.py\nExport: artifacts/07-baseline-overflight.mp4\n'

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=d:\code\sem-proj-asc\backend


In [3]:
import importlib

import s01_utils.baseline_overflight as bof

importlib.reload(bof)

setup = bof.build_baseline_overflight_setup()
bof.print_baseline_setup_summary(setup)

Baseline overflight setup
  targets:           50
  clouds:            27 (seeded over target corridor)
  lead margin:       20.0° before target engage
  altitude:          548.2 km
  orbit window:      None° .. None° (auto if unset)
  capture budget:    10/orbit


In [4]:
rollout = bof.run_baseline_overflight_rollout(setup, show_progress=True)
print(f"steps={rollout.series.t_s.shape[0]}  shutter_cmds={len(rollout.cmd_steps)}")

d:\code\sem-proj-asc\backend\simulation\stepper.py:169: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(
baseline overflight: 100%|██████████| 2947/2947 [01:18<00:00, 37.47it/s]

steps=2948  shutter_cmds=50


In [5]:
kpis = bof.evaluate_baseline_capture_results(rollout.series, rollout.cmd_steps)
bof.print_baseline_capture_kpis(kpis, rollout=rollout)

Baseline capture KPIs
  shutter commands:  50 / 50 targets
  captures taken:    10 / 10 budget
  latent capture:    169.93  (k * cov * quality * (1 - cloud))
  applied capture:   169.93  (latent scaled by cov; 0 if not visible / not taken / repeat target)
  mean quality:      0.4143
  budget exhausted at shutter index: 10
  attitude safety events: 0
  mean sim reward:   -71.639

  cmd  cap  taken  visible  cov    quality  cloud   latent  applied  budget_left
  986  986  yes     yes      0.940   0.9249  0.062    81.50    81.50    9
  1006  1006  yes      no      0.000   0.4658  0.000     0.00     0.00    8
  1026  1026  yes      no      0.000   0.3739  1.000     0.00     0.00    7
  1046  1046  yes      no      0.000   0.3508  1.000     0.00     0.00    6
  1065  1065  yes     yes      0.600   0.3356  0.000    20.14    20.14    5
  1085  1085  yes      no      0.000   0.3373  1.000     0.00     0.00    4
  1105  1105  yes     yes      0.470   0.3398  0.000    15.97    15.97    3
  1125 

In [6]:
import matplotlib

matplotlib.use("Agg")

ARTIFACT_DIR = backend_root / "notebooks" / "s01" / "artifacts"
reward_trace = bof.build_baseline_latent_reward_trace(rollout.series, kpis.capture_results)
video_path = bof.export_baseline_overflight_video(
    rollout.series,
    reward_trace,
    ARTIFACT_DIR / "07-baseline-overflight.mp4",
    shutter_cmd_steps=rollout.cmd_steps,
)

[video] archived previous export -> D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\video_archive\012-07-baseline-overflight.mp4


Writing video: 100%|██████████| 1311/1311 [03:31<00:00,  6.19frame/s]


[mpo_video:after_export] 07-baseline-overflight.mp4 (3253651 bytes, codec=h264)
[mpo_video:play] 07-baseline-overflight.mp4 (3253651 bytes, codec=h264)


artifact=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\07-baseline-overflight.mp4
